<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">


# Python for Finance, 3rd Edition
## Chapter 11 · Performance Python
&copy; Dr. Yves J. Hilpisch<br>
AI-supported by various LLMs<br>
The Python Quants GmbH | https://tpq.io<br>
https://hilpisch.com | https://linktr.ee/dyjh


## Notebook Goals
This notebook mirrors the code examples from the chapter in a Colab-ready
format so that you can run, tweak, and extend them interactively.


### How to Use This Notebook
- Run the cells top to bottom the first time to create all variables.
- Use additional cells for your own experiments or GenAI-assisted
  refactorings.
- Refer back to the book text for detailed explanations and context.


In [ ]:
from pathlib import Path
import subprocess
import sys

NOTEBOOK_SUBDIR = "notebooks"
COLAB_PACKAGES = {"numba": "numba", "Cython": "Cython"}
REPO_NAME = "py4fi3rd"
REPO_URL = "https://github.com/yhilpisch/py4fi3rd.git"


def _support_dir() -> Path:
    cwd = Path.cwd().resolve()
    for candidate in (cwd, *cwd.parents):
        support_dir = candidate / "notebooks"
        if (support_dir / "_book_notebook_support.py").exists():
            return support_dir
    if "google.colab" in sys.modules:
        root = Path("/content") / REPO_NAME
        if not root.exists():
            subprocess.run(
                ["git", "clone", "--depth", "1", REPO_URL, str(root)],
                check=True,
            )
        return root / "notebooks"
    raise RuntimeError("Could not locate notebook support helpers.")


SUPPORT_DIR = _support_dir()
if str(SUPPORT_DIR) not in sys.path:
    sys.path.insert(0, str(SUPPORT_DIR))

from _book_notebook_support import setup_notebook

CONTEXT = setup_notebook(
    notebook_subdir=NOTEBOOK_SUBDIR,
    colab_packages=COLAB_PACKAGES,
)

PROJECT_ROOT = CONTEXT["PROJECT_ROOT"]
NOTEBOOK_DIR = CONTEXT["NOTEBOOK_DIR"]
CODE_DIR = CONTEXT["CODE_DIR"]
CHAPTERS_DIR = CONTEXT["CHAPTERS_DIR"]
FIGURES_DIR = CONTEXT["FIGURES_DIR"]
DATA_DIR = CONTEXT["DATA_DIR"]

PROJECT_ROOT

Performance is often cited as a weakness of Python, especially when people
think of it as a language that only runs slow, interpreted loops. In practice,
modern Python gives you access to JIT compilers, static compilation,
vectorized libraries, and multiprocessing, all from the same code base. This
chapter shows how to combine these tools so that numerically intensive finance
code runs close to compiled-language speed.


## Measuring and Thinking About Performance


Before you optimize any code, you need a simple, repeatable way of measuring
how long a piece of code takes.


## Loops and Vectorization


The prejudice that “Python is slow” usually comes from code that spends
most of its time in Python-level loops.


### Averaging Random Numbers: Pure Python


As a first example, consider a function that draws a large number of
standard-normal random numbers and returns their average.


In [ ]:
import math
import random
def average_py(n: int) -> float:
    acc = 0.0
    for _ in range(n):
        acc += random.gauss(0.0, 1.0)
    return acc / n
# A million Python-level loop iterations with `random.gauss()` clearly show the
# overhead of the interpreter.
%timeit average_py(1_000_000)

### List Comprehensions and Built-in sum()


You can often simplify loop-based functions by replacing the explicit loop
with a list comprehension and the built-in `.sum()` or `sum()` function.


In [ ]:
def average_list(n: int) -> float:
    samples = [random.gauss(0.0, 1.0) for _ in range(n)]
    return sum(samples) / n
%timeit average_list(1_000_000)

### NumPy Vectorization


As soon as you can work with `NumPy` arrays, vectorization usually gives you
the biggest single performance win.


In [ ]:
import numpy as np
rng = np.random.default_rng(seed=42)
def average_np(n: int) -> float:
    samples = rng.standard_normal(n)
    return float(samples.mean())
# Using `NumPy` reduces the run time by roughly two orders of magnitude
# compared to
# `average_py()` on a typical 2026 laptop.
%timeit average_np(1_000_000)

### JIT Compilation with Numba


`Numba` lets you keep the Python structure of your code while compiling the
loop body to efficient machine code using LLVM.


In [ ]:
import numba as nb
@nb.njit
def average_nb(n: int) -> float:
    acc = 0.0
    for i in range(n):
        acc += np.random.standard_normal()
    return acc / n
# Trigger compilation
average_nb(10)
# The `Numba`-compiled loop avoids a large intermediate array, but this
# benchmark is slower than the vectorized `average_np()` on this machine.
%timeit average_nb(1_000_000)

### Simple Multiprocessing


If a task is embarrassingly parallel—for example, you need to compute
independent Monte Carlo scenarios—you can distribute the work across multiple
CPU cores with the `multiprocessing` module.


In [ ]:
import multiprocessing as mp

def average_chunk(n: int, seed: int) -> float:
    local_rng = np.random.default_rng(seed)
    samples = local_rng.standard_normal(n)
    return float(samples.mean())

def average_mp(total_n: int, n_workers: int = 4) -> float:
    chunk = total_n // n_workers
    seeds = list(range(10_000, 10_000 + n_workers))
    if sys.platform == "win32":
        ctx = mp.get_context("spawn")
    else:
        ctx = mp.get_context("fork")
    with ctx.Pool(processes=n_workers) as pool:
        means = pool.starmap(
            average_chunk,
            [(chunk, seed) for seed in seeds],
        )
    return float(np.mean(means))

average_mp(100_000, n_workers=2)


### Cython: Static Compilation of the Loop


`Cython` combines Python syntax with optional C-level type declarations and
then compiles the result to an extension module.


In [ ]:
%load_ext Cython

In [ ]:
%%cython -a
# Import the standard-library `random` module inside the `Cython`
# compilation unit.
import random
# Declare C-level integer and float variables so the compiler can generate
# tight loops.
def average_cy1(int n):
    cdef int i
    cdef float s = 0.0
    for i in range(n):
        s += random.random()
    return s / n

In [ ]:
%%cython
# Import `rand()` from the C standard library.
from libc.stdlib cimport rand
# Access `INT_MAX` from `<limits.h>` so you can normalize the raw integer
# output to
# the unit interval.
cdef extern from "limits.h":
    int INT_MAX
def average_cy2(int n):
    cdef int i
    cdef float s = 0.0
    for i in range(n):
        # Replace the Python-level `random.random()` call with a
        # fast C-level random
        # number generator.
        s += rand() / INT_MAX
    return s / n

## Classic Algorithms and Microbenchmarks


To compare performance across implementations it is useful to look at classic
algorithms that are simple to understand but sensitive to implementation
details.


### Fibonacci Numbers: Recursive vs Iterative


Fibonacci numbers provide a compact benchmark for comparing recursive, cached,
iterative, and compiled implementations.


In [ ]:
def fib_rec_py(n: int) -> int:
    # Handle the base cases of the recursion for `n` equal to `0` or `1`.
    if n < 2:
        return n
    # Express the general case in terms of the two preceding Fibonacci numbers.
    return fib_rec_py(n - 1) + fib_rec_py(n - 2)
# `%timeit` reveals how quickly the naive recursive version slows down as
# `n` grows.
%timeit fib_rec_py(30)

In [ ]:
from functools import cache
@cache
def fib_rec_cached(n: int) -> int:
    # The decorator memoizes the function so each value is computed only once.
    if n < 2:
        return n
    # The recursive relation itself is unchanged; only caching improves
    # performance.
    return fib_rec_cached(n - 1) + fib_rec_cached(n - 2)
# With caching, the complexity becomes linear in _n_ and computing
# `fib_rec_cached(200)` is essentially instant.
%timeit fib_rec_cached(200)

In [ ]:
def fib_it_py(n: int) -> int:
    # Track the current and next Fibonacci numbers in two variables.
    a, b = 0, 1
    # Update both numbers iteratively instead of using recursion.
    for _ in range(n):
        a, b = b, a + b
    # Return the final value after `n` updates.
    return a
# Even in pure Python, the iterative algorithm is faster than the cached
# recursion.
%timeit fib_it_py(200)

In [ ]:
@nb.njit
def fib_it_nb(n: int) -> int:
    # Use plain integers so `Numba` can infer efficient native types.
    a = 0
    b = 1
    # The loop body mirrors the pure Python version but will run as
    # compiled code.
    for _ in range(n):
        a, b = b, a + b
    # Return the final Fibonacci number as before.
    return a
# Trigger compilation
fib_it_nb(10)
# JIT compilation reduces run time by roughly thirty times compared to
# `fib_it_py()`.
%timeit fib_it_nb(90)

### Estimating Pi with Monte Carlo


A classic Monte Carlo exercise is to estimate pi by throwing random points
into a square and counting how many fall into the unit circle.


In [ ]:
import matplotlib.pyplot as plt
n_points = 250_000
# Draw random points uniformly from the square [–1, 1] × [–1, 1].
pts = rng.uniform(low=-1.0, high=1.0, size=(n_points, 2))
radii2 = pts[:, 0]**2 + pts[:, 1]**2
# A point falls inside the unit circle if the squared radius is less than or
# equal to
# 1.
inside = radii2 <= 1.0
pi_estimate = 4.0 * inside.mean()
pi_estimate

In [ ]:
def pi_mc_py(n: int, seed: int = 0) -> float:
    # Use a dedicated `Random` instance so that each run has its own
    # reproducible
    # stream.
    rnd = random.Random(seed)
    # Keep a running count of points inside the unit circle.
    inside = 0
    for _ in range(n):
        x = 2 * rnd.random() - 1
        y = 2 * rnd.random() - 1
        if x * x + y * y <= 1.0:
            inside += 1
    # Estimate pi as four times the fraction of points that fall inside
    # the circle.
    return 4.0 * inside / n
# `%timeit` shows that the pure Python loop quickly becomes the bottleneck.
%timeit pi_mc_py(250_000)
%timeit pi_mc_py(1_000_000)

In [ ]:
@nb.njit
def pi_mc_nb(n: int, seed: int = 0) -> float:
    # Use an integer accumulator just as in the Python version.
    inside = 0
    # Seed NumPy's RNG inside the compiled function for reproducible runs.
    np.random.seed(seed)
    for _ in range(n):
        x = 2.0 * np.random.random() - 1.0
        y = 2.0 * np.random.random() - 1.0
        if x * x + y * y <= 1.0:
            inside += 1
    # The estimator for pi is mathematically identical to the pure
    # Python version.
    return 4.0 * inside / n
# Trigger compilation
pi_mc_nb(10)
# Most of the speedup comes from running the loop body as compiled machine code.
%timeit pi_mc_nb(1_000_000)

## Binomial Option Pricing


The binomial option pricing model is a classic example of an algorithm
that naturally uses nested loops.


### Pure Python Implementation


In a recombining binomial tree, each step moves the price up by a factor _u_
or down by a factor _d_.


In [ ]:
def binomial_price_py(
    s0: float,
    k: float,
    r: float,
    sigma: float,
    t: float,
    steps: int,
    is_call: bool = True,
) -> float:
    # The time step is the maturity divided by the number of binomial steps.
    dt = t / steps
    u = math.exp(sigma * math.sqrt(dt))
    d = 1.0 / u
    # `disc` is the per-step discount factor used in backward induction.
    disc = math.exp(-r * dt)
    # `p` is the risk-neutral probability of an up move.
    p = (math.exp(r * dt) - d) / (u - d)
    # terminal prices
    # Compute all possible terminal prices at maturity in one list
    # comprehension.
    prices = [s0 * (u ** j) * (d ** (steps - j)) for j in range(steps + 1)]
    if is_call:
        values = [max(price - k, 0.0) for price in prices]
    else:
        values = [max(k - price, 0.0) for price in prices]
    # backward induction
    # Step backwards through the tree one layer at a time.
    for step in range(steps - 1, -1, -1):
        values = [
            disc * (p * values[j + 1] + (1.0 - p) * values[j])
            for j in range(step + 1)
        ]
    # The remaining value after backward induction is the option price today.
    return values[0]
# Collect model parameters in a dictionary for convenient reuse.
params = dict(s0=100.0, k=100.0, r=0.02, sigma=0.2, t=1.0, steps=1_000)
# `%timeit` measures the run time of pricing a single option with 1,000 steps.
%timeit binomial_price_py(**params)

### Numba-Compiled Binomial Tree


You can convert the lists to `NumPy` arrays and JIT-compile the entire pricing
function with `Numba`.


In [ ]:
@nb.njit
def binomial_price_nb(
    s0: float,
    k: float,
    r: float,
    sigma: float,
    t: float,
    steps: int,
    is_call: bool = True,
) -> float:
    dt = t / steps
    u = math.exp(sigma * math.sqrt(dt))
    d = 1.0 / u
    disc = math.exp(-r * dt)
    p = (math.exp(r * dt) - d) / (u - d)
    # Allocate a contiguous array of terminal prices so `Numba` can
    # compile tight
    # loops over it.
    prices = np.empty(steps + 1, dtype=np.float64)
    for j in range(steps + 1):
        prices[j] = s0 * (u ** j) * (d ** (steps - j))
    values = np.empty_like(prices)
    for j in range(steps + 1):
        if is_call:
            values[j] = max(prices[j] - k, 0.0)
        else:
            values[j] = max(k - prices[j], 0.0)
    # The nested loops for backward induction now run in compiled code,
    # drastically
    # reducing overhead.
    for step in range(steps - 1, -1, -1):
        for j in range(step + 1):
            values[j] = disc * (p * values[j + 1] + (1.0 - p) * values[j])
    # Convert the scalar option value to a plain Python `float` for
    # downstream use.
    return float(values[0])
# Trigger compilation
binomial_price_nb(**params)
# `%timeit` shows a speedup of roughly a factor of 30 compared to the pure
# Python
# tree.
%timeit binomial_price_nb(**params)

### Cython Binomial Tree


You can also implement the binomial tree directly in `Cython`.


In [ ]:
%%cython -a
import numpy as np
cimport cython
from libc.math cimport exp, sqrt
@cython.boundscheck(False)
@cython.wraparound(False)
def binomial_price_cy(
    double s0,
    double k,
    double r,
    double sigma,
    double t,
    int steps,
    bint is_call=True,
):
    cdef int j, step
    cdef double dt = t / steps
    cdef double u = exp(sigma * sqrt(dt))
    cdef double d = 1.0 / u
    cdef double disc = exp(-r * dt)
    cdef double p = (exp(r * dt) - d) / (u - d)
    cdef double[:] prices = np.empty(steps + 1, dtype=np.float64)
    cdef double[:] values = np.empty(steps + 1, dtype=np.float64)
    for j in range(steps + 1):  # terminal prices
        prices[j] = s0 * (u ** j) * (d ** (steps - j))
        if is_call:
            values[j] = prices[j] - k if prices[j] > k else 0.0
        else:
            values[j] = k - prices[j] if prices[j] < k else 0.0
    for step in range(steps - 1, -1, -1):  # backward induction
        for j in range(step + 1):
            values[j] = disc * (p * values[j + 1] + (1.0 - p) * values[j])
    return float(values[0])

## Monte Carlo Option Pricing


Monte Carlo simulation plays a central role in many pricing and risk
applications.


### Vectorized Monte Carlo with NumPy


You first simulate correlated price paths in a fully vectorized way and
compute discounted payoffs.


In [ ]:
def mc_euro_call_np(
    s0: float,
    k: float,
    r: float,
    sigma: float,
    t: float,
    n_paths: int,
    n_steps: int,
) -> float:
    dt = t / n_steps
    drift = (r - 0.5 * sigma**2) * dt
    vol = sigma * math.sqrt(dt)
    shocks = rng.standard_normal((n_steps, n_paths))
    log_returns = drift + vol * shocks
    log_paths = log_returns.cumsum(axis=0)
    s_t = s0 * np.exp(log_paths[-1])
    payoffs = np.maximum(s_t - k, 0.0)
    discount = math.exp(-r * t)
    return float(discount * payoffs.mean())
mc_params = dict(
    s0=100.0, k=100.0, r=0.02, sigma=0.2, t=1.0, n_paths=250_000, n_steps=252
)
%timeit mc_euro_call_np(**mc_params)

### JIT-Compiled Monte Carlo


As before, you can move the loop into compiled code with `Numba`.


In [ ]:
@nb.njit
def mc_euro_call_nb(
    s0: float,
    k: float,
    r: float,
    sigma: float,
    t: float,
    n_paths: int,
    n_steps: int,
) -> float:
    dt = t / n_steps
    drift = (r - 0.5 * sigma**2) * dt
    vol = sigma * math.sqrt(dt)
    discount = math.exp(-r * t)
    acc = 0.0
    for p in nb.prange(n_paths):
        price = s0
        for _ in range(n_steps):
            z = np.random.standard_normal()
            price *= math.exp(drift + vol * z)
        payoff = price - k
        if payoff > 0.0:
            acc += payoff
    return discount * acc / n_paths
# Trigger compilation
mc_euro_call_nb(**mc_params)
%timeit mc_euro_call_nb(**mc_params)

### Cython Monte Carlo


You can also translate the Monte Carlo simulation into `Cython` and add
C-level type declarations.


In [ ]:
%%cython
import numpy as np
cimport numpy as np
cimport cython
from libc.math cimport exp, sqrt
@cython.boundscheck(False)
@cython.wraparound(False)
def mc_euro_call_cy(
    double s0,
    double k,
    double r,
    double sigma,
    double t,
    int n_paths,
    int n_steps,
):
    cdef int p, step
    cdef double dt = t / n_steps
    cdef double drift = (r - 0.5 * sigma * sigma) * dt
    cdef double vol = sigma * sqrt(dt)
    cdef double discount = exp(-r * t)
    cdef double acc = 0.0
    cdef double price, payoff, z
    cdef double[:, :] rn = np.random.standard_normal((n_steps, n_paths))
    for p in range(n_paths):
        price = s0
        for step in range(n_steps):
            z = rn[step, p]
            price *= exp(drift + vol * z)
        payoff = price - k
        if payoff > 0.0:
            acc += payoff
    return discount * acc / n_paths

In [ ]:
# Benchmark the `Cython` Monte Carlo implementation against the vectorized
# and `Numba` variants.
%timeit mc_euro_call_cy(**mc_params)

## Recursive EWMA on Time Series


An exponentially weighted moving average (EWMA) is defined recursively and is
widely used in risk management and volatility estimation.


### Naive Python Loop


You start with a direct translation of this recursion into Python.


In [ ]:
import pandas as pd
n_obs = 1_000_000
rets = rng.standard_normal(n_obs) * 0.01
lam = 0.94
def ewma_py(x: np.ndarray, lam: float) -> np.ndarray:
    out = np.empty_like(x)
    var = x[0] ** 2
    out[0] = math.sqrt(var)
    for t in range(1, x.shape[0]):
        var = lam * var + (1.0 - lam) * x[t] ** 2
        out[t] = math.sqrt(var)
    return out
%timeit ewma_py(rets, lam)

### Vectorized pandas EWMA


`pandas` provides an EWMA implementation via `.ewm()` that is implemented in C
and optimized for time-series data.


In [ ]:
s = pd.Series(rets)
%timeit s.ewm(alpha=(1 - lam)).std(bias=False)

### Numba-Accelerated EWMA


Finally, you can JIT-compile the loop with `Numba` to reach speeds comparable
to or better than the built-in EWMA while keeping the explicit recursion
visible.


In [ ]:
@nb.njit
def ewma_nb(x: np.ndarray, lam: float) -> np.ndarray:
    out = np.empty_like(x)
    var = x[0] ** 2
    out[0] = math.sqrt(var)
    for t in range(1, x.shape[0]):
        var = lam * var + (1.0 - lam) * x[t] ** 2
        out[t] = math.sqrt(var)
    return out
# Trigger compilation
ewma_nb(rets, lam)[:5]
%timeit ewma_nb(rets, lam)

### Cython EWMA


As a final comparison, you can also implement the EWMA recursion in `Cython`.


In [ ]:
%%cython
import numpy as np
cimport cython
@cython.boundscheck(False)
@cython.wraparound(False)
def ewma_cy(double[:] x, double lam):
    cdef int i
    cdef double var = x[0] * x[0]
    cdef double[:] out = np.empty_like(x)
    out[0] = var ** 0.5
    for i in range(1, x.shape[0]):
        var = lam * var + (1.0 - lam) * x[i] * x[i]
        out[i] = var ** 0.5
    return out

In [ ]:
# Benchmark the `Cython` EWMA recursion against the Python, `pandas`, and
# `Numba` implementations.
%timeit ewma_cy(rets, lam)

<img src="https://hilpisch.com/tpq_logo_bic.png" width="20%" align="right">
